# LLMTIME Preprocessing Implementation

This notebook implements the LLMTIME preprocessing scheme for the Lotka-Volterra predator-prey time series data as described in the coursework. The preprocessing scheme is based on the paper by Gruver et al. (2023).

## Overview

The LLMTIME preprocessing scheme converts numerical time series data into a text-based format suitable for language model tokenization and forecasting.



1. Load the time series data
2. Scale and round the values to a standardized range
3. Format the data as text using a specific convention
4. Tokenize the formatted text for input to the language model

### Data Types

- **Raw trajectories:** `numpy.ndarray` of shape `(n_trajectories, n_timesteps, n_variables)`
    The original numerical time series data.

- **Scaled trajectories:** `numpy.ndarray` with same shape as raw trajectories
    Values scaled (divided by `scaling_factor`) and rounded to specific decimal places.

- **Formatted time series:** 
    * Single timestep: str (e.g., `"1.01,2.06"` for 2 variables)
    * Single time series: str (e.g., `"1.01,2.06;1.02,2.05"` for 2 timesteps)
    * Batch of time series: `List[str]`

- **Tokenized time series:**
    * Single tokenized series: `List[int]` (token IDs from language model tokenizer)
    * Batch of tokenized series: `List[List[int]]`

### Processing Pipeline

1. Load time series data (`LLMTIMEPreprocessor.load_data`): 
    Raw numpy arrays from HDF5 format -> trajectories, time_points

2. Preprocess numerical values (`LLMTIMEPreprocessor.scale_and_round`): 
    Raw trajectories -> Scaled trajectories

3. Format as text (`LLMTIMEPreprocessor.format_*` methods):
    Scaled trajectories -> Formatted time series strings
    * format_timestep: Single timestep array -> Comma-separated string
    * format_timeseries: Single time series -> Semicolon-separated string of timesteps
    * format_timeseries_batch: Multiple time series -> List of formatted strings
    * format_timeseries_slice: Subset of a time series -> Formatted string

4. Tokenization (`LLMTIMEPreprocessor.tokenize_*` methods):
    Formatted strings -> Token IDs for language model input
    * tokenize_sample: Single formatted string -> List of token IDs
    * batch_tokenize: List of formatted strings -> List of token ID lists

5. Reverse operations:
    * `LLMTIMEPreprocessor.decode_sample`: Token IDs -> Formatted string
    * `LLMTIMEPreprocessor.parse_formatted_timeseries`: Formatted string -> Numerical array

Format Convention:
- Variables at the same timestep are separated by commas `(,)`
- Different timesteps are separated by semicolons `(;)`
- Example: `"0.25,1.50;0.27,1.47"` represents 2 timesteps with 2 variables each


Let's implement each of these steps.

In [10]:
import sys
import os
import numpy as np
import matplotlib.pyplot as plt
import h5py
from typing import Tuple, List, Dict, Optional, Union
import logging
from tqdm import tqdm
from transformers import AutoTokenizer

# Add the src directory to the path
sys.path.append('..')
from src.preprocessor import LLMTIMEPreprocessor

# Set up logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(name)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

## 1. Load and Split Data

First, let's load the time series data, and split it into training and test sets.

Throughout the entire project, we will use the **training set** for data preprocessing and model training, and the **test set** to evaluate both the baseline and our model.

In [11]:
# Path to the data file
data_file = '../lotka_volterra_data.h5'

# Load the data
trajectories, time_points = LLMTIMEPreprocessor.load_data(data_file)

print(f"Loaded {trajectories.shape[0]} trajectories with {trajectories.shape[1]} timesteps and {trajectories.shape[2]} variables")
print(f"Time points shape: {time_points.shape}")

2025-03-24 14:36:51,375 - src.preprocessor - INFO - Loaded data with shape: (1000, 100, 2)


Loaded 1000 trajectories with 100 timesteps and 2 variables
Time points shape: (100,)


In [12]:
# Split the trajectories into training and test sets
test_split = 0.2
random_state = 42

train_trajectories, test_trajectories = LLMTIMEPreprocessor.split_train_val(
    trajectories, val_split=test_split, random_state=random_state
)

print(f"Training set: {train_trajectories.shape}, Test set: {test_trajectories.shape}")

2025-03-24 14:36:51,383 - src.preprocessor - INFO - Split data into 800 training and 200 validation trajectories


Training set: (800, 100, 2), Test set: (200, 100, 2)


## 2. Scale and Round the Data

Next, we'll scale the data to a standardized range and round to a fixed number of decimal places. This helps control the token length and ensures consistent representation.

The choice is based on the discussion in `00_data_exploration.ipynb`.

In [13]:
pct_99 = np.percentile(train_trajectories, 99)
scaling_factor = pct_99 / 10    # data = data / scaling_factor
decimal_places = 2  # Number of decimal places to round to

# Scale and round the trajectories
scaled_train_trajectories = LLMTIMEPreprocessor.scale_and_round(train_trajectories, scaling_factor, decimal_places)
scaled_test_trajectories = LLMTIMEPreprocessor.scale_and_round(test_trajectories, scaling_factor, decimal_places)

print(f"Original first trajectory first timestep: {train_trajectories[0, 0]}")
print(f"Scaled first trajectory first timestep: {scaled_train_trajectories[0, 0]}")

2025-03-24 14:36:51,397 - src.preprocessor - INFO - Scaled trajectories by factor 0.6006771326065063 and rounded to 2 decimal places
2025-03-24 14:36:51,397 - src.preprocessor - INFO - Scaled trajectories by factor 0.6006771326065063 and rounded to 2 decimal places


Original first trajectory first timestep: [1.0255829 1.1565652]
Scaled first trajectory first timestep: [1.71 1.93]


## 3. Format the Data as Text

Now, we'll format the scaled data as text using the LLMTIME convention:
- Variables at the same timestep are separated by commas (,)
- Different timesteps are separated by semicolons (;)

In [14]:
# Format a single timestep
timestep_example = scaled_train_trajectories[0, 0]  # First trajectory, first timestep
formatted_timestep = LLMTIMEPreprocessor.format_timestep(timestep_example, decimal_places)
print(f"Formatted timestep: {formatted_timestep}")

# Format a single time series
timeseries_example = scaled_train_trajectories[0]  # First trajectory
formatted_timeseries = LLMTIMEPreprocessor.format_timeseries(timeseries_example, decimal_places)
print(f"Formatted time series (first 50 chars): {formatted_timeseries[:50]}...")

# Format a batch of time series (just a few for demonstration)
batch_example = scaled_train_trajectories[:5]  # First 5 trajectories
formatted_batch = LLMTIMEPreprocessor.format_timeseries_batch(batch_example, decimal_places)
print(f"Formatted batch size: {len(formatted_batch)}")
print(f"First formatted series in batch (first 50 chars): {formatted_batch[0][:50]}... \
      \nSecond formatted series in batch (first 50 chars): {formatted_batch[1][:50]}...")

2025-03-24 14:36:51,506 - src.preprocessor - INFO - Formatted 5 time series into LLMTIME format


Formatted timestep: 1.71,1.93
Formatted time series (first 50 chars): 1.71,1.93;1.54,1.71;1.46,1.49;1.45,1.29;1.52,1.12;...
Formatted batch size: 5
First formatted series in batch (first 50 chars): 1.71,1.93;1.54,1.71;1.46,1.49;1.45,1.29;1.52,1.12;...       
Second formatted series in batch (first 50 chars): 1.54,1.95;0.91,1.29;0.70,0.80;0.65,0.49;0.68,0.30;...


## 4. Tokenize the Formatted Text

Finally, we'll tokenize the formatted text using the Qwen2.5-Instruct tokenizer.

In [15]:
# Load the tokenizer
tokenizer_name = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(tokenizer_name)

# Tokenize a single formatted time series
tokenized_example = LLMTIMEPreprocessor.tokenize_sample(formatted_timeseries, tokenizer)
print(f"Tokenized time series length: {len(tokenized_example)}")
print(f"First 20 tokens: {tokenized_example[:20]}")

# Decode the tokens back to text to verify
decoded_example = LLMTIMEPreprocessor.decode_sample(tokenized_example, tokenizer)
print(f"Decoded time series (first 50 chars): {decoded_example[:50]}...")

Tokenized time series length: 999
First 20 tokens: [16, 13, 22, 16, 11, 16, 13, 24, 18, 26, 16, 13, 20, 19, 11, 16, 13, 22, 16, 26]
Decoded time series (first 50 chars): 1.71,1.93;1.54,1.71;1.46,1.49;1.45,1.29;1.52,1.12;...


## 5. Example Sequences

Let's demonstrate two complete examples of the preprocessing pipeline, from raw data to tokenized sequences.

In [16]:
def demonstrate_preprocessing_pipeline(trajectory_idx, n_timesteps=10):
    """
    Demonstrate the complete preprocessing pipeline for a single trajectory.
    
    Args:
        trajectory_idx (int): Index of the trajectory to use.
        n_timesteps (int): Number of timesteps to include in the example.
    """
    print(f"\n--- Example {trajectory_idx + 1} ---")
    
    # 1. Extract a subset of the raw trajectory
    raw_trajectory = trajectories[trajectory_idx, :n_timesteps]
    print(f"Raw trajectory shape: {raw_trajectory.shape}")
    print(f"Raw trajectory:\n{raw_trajectory}")
    
    # 2. Scale and round
    scaled_trajectory = LLMTIMEPreprocessor.scale_and_round(raw_trajectory, scaling_factor, decimal_places)
    print(f"\nScaled trajectory:\n{scaled_trajectory}")
    
    # 3. Format as text
    formatted_text = LLMTIMEPreprocessor.format_timeseries(scaled_trajectory, decimal_places)
    print(f"\nFormatted text:\n{formatted_text}")
    
    # 4. Tokenize
    tokenized = LLMTIMEPreprocessor.tokenize_sample(formatted_text, tokenizer)
    print(f"\nTokenized sequence (length {len(tokenized)}):\n{tokenized}")
    
    # 5. Map tokens to their string representations
    token_strings = [tokenizer.decode([token]) for token in tokenized]
    print(f"\nToken strings:\n{token_strings}")
    
    return {
        'raw': raw_trajectory,
        'scaled': scaled_trajectory,
        'formatted': formatted_text,
        'tokenized': tokenized,
        'token_strings': token_strings
    }

# Demonstrate with two different examples
example1 = demonstrate_preprocessing_pipeline(0, n_timesteps=5)  # First trajectory
example2 = demonstrate_preprocessing_pipeline(10, n_timesteps=5)  # Another trajectory

2025-03-24 14:36:51,941 - src.preprocessor - INFO - Scaled trajectories by factor 0.6006771326065063 and rounded to 2 decimal places
2025-03-24 14:36:51,943 - src.preprocessor - INFO - Scaled trajectories by factor 0.6006771326065063 and rounded to 2 decimal places



--- Example 1 ---
Raw trajectory shape: (5, 2)
Raw trajectory:
[[0.94991744 1.040624  ]
 [0.74055135 0.7795419 ]
 [0.6822457  0.56439036]
 [0.7166742  0.40764433]
 [0.82451135 0.30028325]]

Scaled trajectory:
[[1.58 1.73]
 [1.23 1.3 ]
 [1.14 0.94]
 [1.19 0.68]
 [1.37 0.5 ]]

Formatted text:
1.58,1.73;1.23,1.30;1.14,0.94;1.19,0.68;1.37,0.50

Tokenized sequence (length 49):
[16, 13, 20, 23, 11, 16, 13, 22, 18, 26, 16, 13, 17, 18, 11, 16, 13, 18, 15, 26, 16, 13, 16, 19, 11, 15, 13, 24, 19, 26, 16, 13, 16, 24, 11, 15, 13, 21, 23, 26, 16, 13, 18, 22, 11, 15, 13, 20, 15]

Token strings:
['1', '.', '5', '8', ',', '1', '.', '7', '3', ';', '1', '.', '2', '3', ',', '1', '.', '3', '0', ';', '1', '.', '1', '4', ',', '0', '.', '9', '4', ';', '1', '.', '1', '9', ',', '0', '.', '6', '8', ';', '1', '.', '3', '7', ',', '0', '.', '5', '0']

--- Example 11 ---
Raw trajectory shape: (5, 2)
Raw trajectory:
[[0.96647114 1.094717  ]
 [0.75811523 0.94788396]
 [0.66393834 0.7895862 ]
 [0.6488795  0.6488567 ]


## 6. Save Preprocessed Data

Finally, let's save the preprocessed data for later use in training and evaluation.

In [17]:
import pickle

formatted_train = LLMTIMEPreprocessor.format_timeseries_batch(scaled_train_trajectories, decimal_places)
formatted_test = LLMTIMEPreprocessor.format_timeseries_batch(scaled_test_trajectories, decimal_places)
tokenized_train = LLMTIMEPreprocessor.batch_tokenize(formatted_train)
tokenized_test = LLMTIMEPreprocessor.batch_tokenize(formatted_test)

# Create a directory to save the preprocessed data
os.makedirs('../data', exist_ok=True)

# Save the training and validation sets
with open('../data/scaled_train.pkl', 'wb') as f:
    pickle.dump(scaled_train_trajectories, f)

with open('../data/scaled_test.pkl', 'wb') as f:
    pickle.dump(scaled_test_trajectories, f)

with open('../data/formatted_train.pkl', 'wb') as f:
    pickle.dump(formatted_train, f)

with open('../data/formatted_test.pkl', 'wb') as f:
    pickle.dump(formatted_test, f)

with open('../data/tokenized_train.pkl', 'wb') as f:
    pickle.dump(tokenized_train, f)

with open('../data/tokenized_test.pkl', 'wb') as f:
    pickle.dump(tokenized_test, f)

print("Preprocessed data saved to disk.")

2025-03-24 14:36:52,149 - src.preprocessor - INFO - Formatted 800 time series into LLMTIME format
2025-03-24 14:36:52,172 - src.preprocessor - INFO - Formatted 200 time series into LLMTIME format
Tokenizing samples: 100%|██████████| 200/200 [00:00<00:00, 1028.92it/s]


Preprocessed data saved to disk.


In [18]:
# Load the preprocessed data to verify
with open('../data/scaled_train.pkl', 'rb') as f:
    scaled_train_trajectories = pickle.load(f)

with open('../data/scaled_test.pkl', 'rb') as f:
    scaled_test_trajectories = pickle.load(f)

with open('../data/formatted_train.pkl', 'rb') as f:
    formatted_train = pickle.load(f)

with open('../data/formatted_test.pkl', 'rb') as f:
    formatted_test = pickle.load(f)

with open('../data/tokenized_train.pkl', 'rb') as f:
    tokenized_train = pickle.load(f)

with open('../data/tokenized_test.pkl', 'rb') as f:
    tokenized_test = pickle.load(f)

print(f"Formatted train data size: {len(formatted_train)}")
print(f"First formatted series in batch (first 50 chars): {formatted_train[0][:50]}... \
      \nSecond formatted series in batch (first 50 chars): {formatted_train[1][:50]}...")

decoded_train_example = LLMTIMEPreprocessor.decode_sample(tokenized_train[0], tokenizer)
print(f"Decoded time series (first 50 chars): {decoded_train_example[:50]}...")

Formatted train data size: 800
First formatted series in batch (first 50 chars): 1.71,1.93;1.54,1.71;1.46,1.49;1.45,1.29;1.52,1.12;...       
Second formatted series in batch (first 50 chars): 1.54,1.95;0.91,1.29;0.70,0.80;0.65,0.49;0.68,0.30;...
Decoded time series (first 50 chars): 1.71,1.93;1.54,1.71;1.46,1.49;1.45,1.29;1.52,1.12;...
